# Defects4J + EvoSuite: Resumable Java Test Generation Experiment

This notebook runs EvoSuite on the Easy, Medium, and Hard Java datasets.

**Experiment design**
- 10 methods per difficulty level.
- 5 independent repetitions per method.
- 50 runs per level and 150 runs in total.
- Defects4J buggy versions are checked out for each repetition.
- EvoSuite search budget: 60 seconds per run.
- Generation success is recorded when at least one `*_ESTest.java` file is created. This does not mean that the generated tests have been compiled or executed.

**Important:** Existing checkpoints are preserved by default. Completed and failed runs are not repeated unless `RETRY_FAILED` is enabled. The notebook exports both Excel summaries and the raw checkpoint data.

Run the cells from top to bottom in Google Colab.

## 1. Upload the datasets and, optionally, a previous checkpoint archive

Upload the three Java Excel files. To resume an earlier experiment in a fresh Colab session, you may also upload `EvoSuite_Java_150_Complete_Results.zip`.

Expected dataset filenames (a Colab duplicate suffix such as `(1)` is accepted):
- `Java_easy_dataset_cleaned.xlsx`
- `Java_midum_dataset.xlsx`
- `Java_Hard_dataset_cleaned.xlsx`

In [ ]:
from pathlib import Path
from google.colab import files

DATASET_STEMS = {
    "Easy": "Java_easy_dataset_cleaned",
    "Medium": "Java_midum_dataset",
    "Hard": "Java_Hard_dataset_cleaned",
}


def find_uploaded_dataset(stem):
    exact = Path("/content") / f"{stem}.xlsx"
    if exact.exists():
        return exact

    matches = list(Path("/content").glob(f"{stem}*.xlsx"))
    if not matches:
        return None

    matches.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    return matches[0]


missing_levels = [
    level for level, stem in DATASET_STEMS.items()
    if find_uploaded_dataset(stem) is None
]

if missing_levels:
    print("Upload the missing Excel datasets. You may also select a previous checkpoint ZIP.")
    uploaded = files.upload()
    print("Uploaded files:")
    for filename in uploaded:
        print(f"- {filename}")
else:
    print("All three dataset files are already available in /content.")

for level, stem in DATASET_STEMS.items():
    path = find_uploaded_dataset(stem)
    if path is None:
        raise FileNotFoundError(
            f"Missing {level} dataset. Expected a file beginning with '{stem}'."
        )
    print(f"{level}: {path}")

## 2. Install system dependencies

The experiment uses Java 11, Maven, Subversion, Git, Perl, and the Perl dependencies required by Defects4J.

In [ ]:
import os
import subprocess
import sys
import time
import json
import hashlib
import shutil
import re
import zipfile
import platform
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd


def run_setup_command(command, cwd=None, env=None, timeout=1800):
    result = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=timeout,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(map(str, command))}\n"
            f"STDOUT:\n{result.stdout[-5000:]}\n"
            f"STDERR:\n{result.stderr[-5000:]}"
        )
    return result


if shutil.which("apt-get"):
    run_setup_command(["apt-get", "update", "-qq"], timeout=1800)
    run_setup_command(
        [
            "apt-get", "install", "-y", "-qq",
            "openjdk-11-jdk", "maven", "subversion", "git",
            "perl", "libdbi-perl", "build-essential", "cpanminus",
            "wget", "unzip",
        ],
        timeout=2400,
    )
else:
    print("apt-get is unavailable. Confirm that the required system dependencies are installed.")

JAVA11 = "/usr/lib/jvm/java-11-openjdk-amd64"
if not Path(JAVA11).exists():
    raise FileNotFoundError(f"Java 11 installation not found at {JAVA11}.")

os.environ["JAVA_HOME"] = JAVA11
os.environ["PATH"] = f"{JAVA11}/bin:{os.environ.get('PATH', '')}"
os.environ["PERL_MM_USE_DEFAULT"] = "1"

java_version = subprocess.run(
    ["java", "-version"], capture_output=True, text=True, errors="replace"
)
print(java_version.stderr or java_version.stdout)
print("Maven:", subprocess.run(["mvn", "-version"], capture_output=True, text=True, errors="replace").stdout.splitlines()[0])
print("Subversion:", subprocess.run(["svn", "--version", "--quiet"], capture_output=True, text=True, errors="replace").stdout.strip())

perl_module_check = subprocess.run(
    ["perl", "-MString::Interpolate", "-e", "1"],
    capture_output=True,
    text=True,
    errors="replace",
)
if perl_module_check.returncode != 0:
    run_setup_command(
        ["cpan", "-T", "String::Interpolate"],
        env=os.environ.copy(),
        timeout=1800,
    )

print("System dependencies are ready.")

## 3. Install or validate Defects4J and locate EvoSuite

The notebook reuses an existing Defects4J installation. If none is available, it clones the repository and runs `init.sh`.

In [ ]:
D4J_DIR = Path("/content/defects4j")
D4J_PATH = D4J_DIR / "framework" / "bin" / "defects4j"

setup_env = os.environ.copy()
setup_env["JAVA_HOME"] = JAVA11
setup_env["PATH"] = f"{JAVA11}/bin:{D4J_DIR / 'framework' / 'bin'}:{setup_env.get('PATH', '')}"
setup_env["PERL_MM_USE_DEFAULT"] = "1"

if not D4J_PATH.exists():
    if D4J_DIR.exists():
        shutil.rmtree(D4J_DIR)
    run_setup_command(
        ["git", "clone", "https://github.com/rjust/defects4j.git", str(D4J_DIR)],
        env=setup_env,
        timeout=1800,
    )

if not D4J_PATH.exists():
    raise FileNotFoundError(f"Defects4J executable not found: {D4J_PATH}")

quick_check = subprocess.run(
    [str(D4J_PATH), "info", "-p", "Codec"],
    cwd="/content",
    env=setup_env,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=300,
)

if quick_check.returncode != 0:
    init_script = D4J_DIR / "init.sh"
    if not init_script.exists():
        raise FileNotFoundError(f"Defects4J initialization script not found: {init_script}")

    init_result = subprocess.run(
        ["bash", str(init_script)],
        cwd=str(D4J_DIR),
        env=setup_env,
        capture_output=True,
        text=True,
        errors="replace",
        timeout=3600,
    )
    if init_result.returncode != 0:
        print(init_result.stdout[-5000:])
        print(init_result.stderr[-5000:])
        raise RuntimeError("Defects4J initialization failed.")

EVOSUITE_CANDIDATES = [
    D4J_DIR / "framework" / "lib" / "test_generation" / "generation" / "evosuite-1.1.0.jar",
    D4J_DIR / "framework" / "lib" / "evosuite-1.1.0.jar",
]

EVOSUITE_JAR = next((path for path in EVOSUITE_CANDIDATES if path.exists()), None)
if EVOSUITE_JAR is None:
    jar_matches = list(D4J_DIR.rglob("evosuite-1.1.0.jar"))
    EVOSUITE_JAR = jar_matches[0] if jar_matches else None

if EVOSUITE_JAR is None:
    raise FileNotFoundError("EvoSuite 1.1.0 JAR was not found inside the Defects4J installation.")

info_result = subprocess.run(
    [str(D4J_PATH), "info", "-p", "Codec"],
    cwd="/content",
    env=setup_env,
    capture_output=True,
    text=True,
    errors="replace",
    timeout=300,
)
if info_result.returncode != 0:
    raise RuntimeError(
        "Defects4J validation failed.\n"
        f"STDOUT:\n{info_result.stdout[-4000:]}\n"
        f"STDERR:\n{info_result.stderr[-4000:]}"
    )

print("Defects4J:", D4J_PATH)
print("EvoSuite:", EVOSUITE_JAR)
print("Defects4J validation passed.")

## 4. Configure the experiment

The settings below preserve the experiment design used in the source notebook. `RESET_CHECKPOINTS` is disabled to prevent accidental loss of completed runs.

In [ ]:
LEVELS = ["Easy", "Medium", "Hard"]
REPETITIONS_PER_PROJECT = 5
EXPECTED_ROWS_PER_LEVEL = 10
SEARCH_BUDGET_SECONDS = 60
MAX_WORKERS = 10

CHECKOUT_TIMEOUT_SECONDS = 600
COMPILE_TIMEOUT_SECONDS = 1200
CLASSPATH_TIMEOUT_SECONDS = 180
EVOSUITE_TIMEOUT_SECONDS = 3600

RETRY_FAILED = False
RESET_CHECKPOINTS = False

BASE = Path("/content/evosuite_experiment")
EXPORT_DIR = BASE / "exports"

DATASET_PATHS = {
    level: find_uploaded_dataset(stem)
    for level, stem in DATASET_STEMS.items()
}

for level in LEVELS:
    (BASE / level / "checkpoints").mkdir(parents=True, exist_ok=True)
    (BASE / level / "runs").mkdir(parents=True, exist_ok=True)
    (BASE / level / "projects").mkdir(parents=True, exist_ok=True)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment directory:", BASE)
print("Levels:", LEVELS)
print("Repetitions per project:", REPETITIONS_PER_PROJECT)
print("Maximum workers:", MAX_WORKERS)
print("EvoSuite search budget:", SEARCH_BUDGET_SECONDS, "seconds")
print("EvoSuite process timeout:", EVOSUITE_TIMEOUT_SECONDS, "seconds")

## 5. Load and validate the datasets

The notebook selects the first non-empty worksheet containing all required Defects4J metadata columns, so empty worksheets are ignored automatically.

In [ ]:
REQUIRED_COLUMNS = [
    "project_id",
    "bug_id",
    "task_id",
    "qualified_class_name",
    "function_name",
]


def load_dataset(path, level):
    workbook = pd.ExcelFile(path)
    selected_sheet = None
    selected_df = None

    for sheet_name in workbook.sheet_names:
        candidate = pd.read_excel(path, sheet_name=sheet_name)
        if len(candidate) > 0 and set(REQUIRED_COLUMNS).issubset(candidate.columns):
            selected_sheet = sheet_name
            selected_df = candidate.copy()
            break

    if selected_df is None:
        raise ValueError(
            f"No populated worksheet in {path.name} contains all required columns: {REQUIRED_COLUMNS}"
        )

    selected_df = selected_df.reset_index(drop=True)

    missing_values = selected_df[REQUIRED_COLUMNS].isna().any()
    invalid_columns = missing_values[missing_values].index.tolist()
    if invalid_columns:
        raise ValueError(f"{level} dataset has missing required values in: {invalid_columns}")

    selected_df["project_id"] = selected_df["project_id"].astype(str).str.strip()
    selected_df["bug_id"] = selected_df["bug_id"].astype(int)
    selected_df["task_id"] = selected_df["task_id"].astype(str)
    selected_df["qualified_class_name"] = selected_df["qualified_class_name"].astype(str).str.strip()
    selected_df["function_name"] = selected_df["function_name"].astype(str).str.strip()

    if EXPECTED_ROWS_PER_LEVEL is not None and len(selected_df) != EXPECTED_ROWS_PER_LEVEL:
        raise ValueError(
            f"{level} contains {len(selected_df)} rows; expected {EXPECTED_ROWS_PER_LEVEL}. "
            "Update EXPECTED_ROWS_PER_LEVEL only if the experiment design has changed."
        )

    print(
        f"{level}: file={path.name}, sheet={selected_sheet}, rows={len(selected_df)}, "
        f"columns={len(selected_df.columns)}"
    )
    return selected_df


data = {
    level: load_dataset(DATASET_PATHS[level], level)
    for level in LEVELS
}

print("Total source methods:", sum(len(frame) for frame in data.values()))
print("Expected total repetitions:", sum(len(frame) for frame in data.values()) * REPETITIONS_PER_PROJECT)

## 6. Restore an optional checkpoint archive

If you uploaded a previous results ZIP, this cell restores its `state.json` files only when the corresponding checkpoint does not already exist. Existing checkpoints are never overwritten automatically.

In [ ]:
from zipfile import ZipFile

uploaded_archives = list(Path("/content").glob("*EvoSuite*Complete*Results*.zip"))

if uploaded_archives:
    archive_path = max(uploaded_archives, key=lambda path: path.stat().st_mtime)
    print("Checkpoint archive detected:", archive_path.name)

    with ZipFile(archive_path, "r") as archive:
        for level in LEVELS:
            target = BASE / level / "checkpoints" / "state.json"
            if target.exists():
                print(f"Preserving existing {level} checkpoint.")
                continue

            candidates = [
                name for name in archive.namelist()
                if name.endswith(f"/{level}/state.json") or name.endswith(f"{level}/state.json")
            ]
            if candidates:
                member = candidates[0]
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, open(target, "wb") as destination:
                    shutil.copyfileobj(source, destination)
                print(f"Restored {level} checkpoint from {member}")
else:
    print("No previous checkpoint ZIP detected. New checkpoints will be initialized if required.")

## 7. Create or validate checkpoints

A checkpoint is created only when missing. If a checkpoint already exists, its row identifiers are checked against the uploaded dataset before it is reused. The five repetition records are preserved.

In [ ]:
def atomic_write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_name(path.name + ".tmp")

    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2)
        handle.flush()
        os.fsync(handle.fileno())

    temp_path.replace(path)


def dataset_signature(frame):
    identity_columns = [
        "project_id", "bug_id", "task_id", "qualified_class_name", "function_name"
    ]
    records = frame[identity_columns].astype(str).to_dict(orient="records")
    serialized = json.dumps(records, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def make_pending_repetition():
    return {
        "status": "pending",
        "test_code": None,
        "test_count": 0,
        "elapsed_seconds": None,
        "evosuite_seconds": None,
        "error": None,
        "return_code": None,
        "stage": None,
        "stdout_tail": None,
        "stderr_tail": None,
        "generated_files": [],
        "generated_file_contents": [],
    }


def expected_checkpoint_rows(frame):
    rows = []
    for source_row, record in frame.iterrows():
        rows.append({
            "source_row": int(source_row),
            "project_id": str(record["project_id"]),
            "bug_id": int(record["bug_id"]),
            "task_id": str(record["task_id"]),
            "qualified_class_name": str(record["qualified_class_name"]),
            "function_name": str(record["function_name"]),
            "batch": int(source_row) + 1,
            "repetitions": {
                str(rep): make_pending_repetition()
                for rep in range(1, REPETITIONS_PER_PROJECT + 1)
            },
        })
    return rows


def initialize_or_validate_checkpoint(level, frame):
    checkpoint_path = BASE / level / "checkpoints" / "state.json"

    if RESET_CHECKPOINTS and checkpoint_path.exists():
        checkpoint_path.unlink()
        print(f"Reset requested: removed {level} checkpoint.")

    signature = dataset_signature(frame)

    if not checkpoint_path.exists():
        state = {
            "schema_version": 2,
            "level": level,
            "dataset_signature": signature,
            "total_projects": len(frame),
            "repetitions_per_project": REPETITIONS_PER_PROJECT,
            "total_runs": len(frame) * REPETITIONS_PER_PROJECT,
            "rows": expected_checkpoint_rows(frame),
        }
        atomic_write_json(checkpoint_path, state)
        print(f"Created checkpoint: {checkpoint_path}")
        return

    with open(checkpoint_path, "r", encoding="utf-8") as handle:
        state = json.load(handle)

    expected_rows = expected_checkpoint_rows(frame)
    existing_rows = state.get("rows", [])

    if len(existing_rows) != len(expected_rows):
        raise ValueError(
            f"{level} checkpoint has {len(existing_rows)} rows, but the dataset has {len(expected_rows)}. "
            "The checkpoint was not changed."
        )

    for expected, existing in zip(expected_rows, existing_rows):
        for key in ["source_row", "project_id", "bug_id", "task_id", "qualified_class_name", "function_name"]:
            if str(existing.get(key)) != str(expected.get(key)):
                raise ValueError(
                    f"{level} checkpoint does not match the uploaded dataset at row {expected['source_row']} "
                    f"(field: {key}). The checkpoint was not changed."
                )

        existing["batch"] = expected["batch"]
        repetitions = existing.setdefault("repetitions", {})
        for rep in range(1, REPETITIONS_PER_PROJECT + 1):
            repetitions.setdefault(str(rep), make_pending_repetition())

    existing_signature = state.get("dataset_signature")
    if existing_signature and existing_signature != signature:
        raise ValueError(f"{level} dataset signature mismatch. The existing checkpoint was preserved.")

    state["schema_version"] = state.get("schema_version", 1)
    state["dataset_signature"] = signature
    state["total_projects"] = len(frame)
    state["repetitions_per_project"] = REPETITIONS_PER_PROJECT
    state["total_runs"] = len(frame) * REPETITIONS_PER_PROJECT
    atomic_write_json(checkpoint_path, state)
    print(f"Validated and preserved checkpoint: {checkpoint_path}")


for level in LEVELS:
    initialize_or_validate_checkpoint(level, data[level])

## 8. Checkpoint utilities

Only the notebook's main process writes checkpoint files. Worker threads return results; the main process saves each completed repetition atomically.

In [ ]:
def read_checkpoint(level):
    checkpoint_path = BASE / level / "checkpoints" / "state.json"
    with open(checkpoint_path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def get_pending_runs(level):
    state = read_checkpoint(level)
    pending = []

    for row in state["rows"]:
        for repetition_key, result in row["repetitions"].items():
            status = result.get("status", "pending")
            should_run = status == "pending" or (RETRY_FAILED and status == "failed")

            if should_run:
                pending.append({
                    "level": level,
                    "source_row": int(row["source_row"]),
                    "project_id": str(row["project_id"]),
                    "bug_id": int(row["bug_id"]),
                    "task_id": str(row["task_id"]),
                    "qualified_class_name": str(row["qualified_class_name"]),
                    "function_name": str(row["function_name"]),
                    "batch": int(row.get("batch", int(row["source_row"]) + 1)),
                    "repetition": int(repetition_key),
                })

    return pending


def update_checkpoint(level, source_row, repetition_no, result):
    checkpoint_path = BASE / level / "checkpoints" / "state.json"
    state = read_checkpoint(level)

    target_row = next(
        (row for row in state["rows"] if int(row["source_row"]) == int(source_row)),
        None,
    )
    if target_row is None:
        raise KeyError(f"Source row {source_row} not found in {level} checkpoint.")

    repetition_key = str(int(repetition_no))
    current = target_row["repetitions"].setdefault(repetition_key, make_pending_repetition())
    current.update(result)
    current["status"] = result.get("status", "failed")

    atomic_write_json(checkpoint_path, state)


def collect_pending_tasks():
    by_level = {level: get_pending_runs(level) for level in LEVELS}
    tasks = []
    max_count = max((len(items) for items in by_level.values()), default=0)

    for index in range(max_count):
        for level in LEVELS:
            if index < len(by_level[level]):
                tasks.append(by_level[level][index])

    return tasks

print("Checkpoint utilities are ready.")

## 9. EvoSuite worker

Each repetition receives its own Defects4J checkout and EvoSuite output directory. The worker does not modify checkpoints. The success criterion remains consistent with the source notebook: at least one generated `*_ESTest.java` file must exist.

In [ ]:
JAVA_ENV = os.environ.copy()
JAVA_ENV["JAVA_HOME"] = JAVA11
JAVA_ENV["PATH"] = f"{JAVA11}/bin:{D4J_DIR / 'framework' / 'bin'}:{JAVA_ENV.get('PATH', '')}"


def text_tail(value, limit=5000):
    if value is None:
        return ""
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    return str(value)[-limit:]


def run_command(command, cwd, env, timeout_seconds):
    try:
        completed = subprocess.run(
            command,
            cwd=str(cwd),
            env=env,
            capture_output=True,
            text=True,
            errors="replace",
            timeout=timeout_seconds,
        )
        return {
            "return_code": completed.returncode,
            "stdout": completed.stdout or "",
            "stderr": completed.stderr or "",
            "timed_out": False,
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "return_code": None,
            "stdout": text_tail(exc.stdout, limit=10000),
            "stderr": text_tail(exc.stderr, limit=10000),
            "timed_out": True,
        }


def count_generated_tests(test_code):
    if not test_code:
        return 0
    return len(re.findall(r"(?m)^\s*@(?:org\.junit\.)?Test\b", test_code))


def make_failure_result(task, started_at, stage, error, return_code=None, stdout="", stderr=""):
    return {
        "source_row": int(task["source_row"]),
        "repetition": int(task["repetition"]),
        "status": "failed",
        "test_code": None,
        "test_count": 0,
        "elapsed_seconds": round(time.time() - started_at, 2),
        "evosuite_seconds": None,
        "return_code": return_code,
        "error": text_tail(error, 5000) if error else None,
        "stage": stage,
        "stdout_tail": text_tail(stdout),
        "stderr_tail": text_tail(stderr),
        "generated_files": [],
    }


def run_one_repetition(task):
    level = task["level"]
    source_row = int(task["source_row"])
    project_id = str(task["project_id"])
    bug_id = int(task["bug_id"])
    class_name = str(task["qualified_class_name"])
    repetition = int(task["repetition"])

    safe_project = re.sub(r"[^A-Za-z0-9_.-]+", "_", project_id)
    run_key = f"row{source_row}_{safe_project}-{bug_id}_rep{repetition}"

    project_dir = BASE / level / "projects" / run_key
    run_dir = BASE / level / "runs" / f"row{source_row}_{safe_project}-{bug_id}" / f"rep{repetition}"

    shutil.rmtree(project_dir, ignore_errors=True)
    shutil.rmtree(run_dir, ignore_errors=True)
    project_dir.mkdir(parents=True, exist_ok=True)
    run_dir.mkdir(parents=True, exist_ok=True)

    env = JAVA_ENV.copy()
    started_at = time.time()

    try:
        checkout = run_command(
            [str(D4J_PATH), "checkout", "-p", project_id, "-v", f"{bug_id}b", "-w", str(project_dir)],
            cwd="/content",
            env=env,
            timeout_seconds=CHECKOUT_TIMEOUT_SECONDS,
        )
        if checkout["timed_out"] or checkout["return_code"] != 0:
            reason = "Defects4J checkout timed out." if checkout["timed_out"] else "Defects4J checkout failed."
            return make_failure_result(
                task, started_at, "checkout", reason,
                checkout["return_code"], checkout["stdout"], checkout["stderr"]
            )

        compile_result = run_command(
            [str(D4J_PATH), "compile"],
            cwd=project_dir,
            env=env,
            timeout_seconds=COMPILE_TIMEOUT_SECONDS,
        )
        if compile_result["timed_out"] or compile_result["return_code"] != 0:
            reason = "Defects4J compilation timed out." if compile_result["timed_out"] else "Defects4J compilation failed."
            return make_failure_result(
                task, started_at, "compile", reason,
                compile_result["return_code"], compile_result["stdout"], compile_result["stderr"]
            )

        classpath_result = run_command(
            [str(D4J_PATH), "export", "-p", "cp.compile"],
            cwd=project_dir,
            env=env,
            timeout_seconds=CLASSPATH_TIMEOUT_SECONDS,
        )
        if classpath_result["timed_out"] or classpath_result["return_code"] != 0:
            reason = "Compile classpath export timed out." if classpath_result["timed_out"] else "Compile classpath export failed."
            return make_failure_result(
                task, started_at, "classpath", reason,
                classpath_result["return_code"], classpath_result["stdout"], classpath_result["stderr"]
            )

        classpath = classpath_result["stdout"].strip()
        if not classpath:
            return make_failure_result(
                task, started_at, "classpath", "Defects4J returned an empty compile classpath.",
                classpath_result["return_code"], classpath_result["stdout"], classpath_result["stderr"]
            )

        evo_started_at = time.time()
        evo = run_command(
            [
                "java", "-jar", str(EVOSUITE_JAR),
                "-class", class_name,
                "-projectCP", classpath,
                f"-Dsearch_budget={SEARCH_BUDGET_SECONDS}",
                f"-Dtest_dir={run_dir}",
            ],
            cwd=project_dir,
            env=env,
            timeout_seconds=EVOSUITE_TIMEOUT_SECONDS,
        )
        evo_elapsed = round(time.time() - evo_started_at, 2)
        total_elapsed = round(time.time() - started_at, 2)

        generated_files = sorted(run_dir.rglob("*_ESTest.java"))
        if generated_files:
            file_contents = []
            relative_paths = []

            for generated_path in generated_files:
                content = generated_path.read_text(encoding="utf-8", errors="replace")
                relative_path = str(generated_path.relative_to(run_dir))
                relative_paths.append(relative_path)
                file_contents.append({"path": relative_path, "content": content})

            primary_test_code = file_contents[0]["content"]
            all_test_source = "\n\n".join(item["content"] for item in file_contents)
            test_count = count_generated_tests(all_test_source)
            warning = None
            if test_count == 0:
                warning = "ESTest file exists, but no JUnit @Test annotation was detected by the counter."
            elif evo["return_code"] not in (0, None):
                warning = f"EvoSuite returned code {evo['return_code']} but generated test source files."

            return {
                "source_row": source_row,
                "repetition": repetition,
                "status": "success",
                "test_code": primary_test_code,
                "generated_file_contents": file_contents,
                "test_count": test_count,
                "elapsed_seconds": total_elapsed,
                "evosuite_seconds": evo_elapsed,
                "return_code": evo["return_code"],
                "error": warning,
                "stage": "generation",
                "stdout_tail": text_tail(evo["stdout"]),
                "stderr_tail": text_tail(evo["stderr"]),
                "generated_files": relative_paths,
            }

        if evo["timed_out"]:
            reason = f"EvoSuite exceeded the {EVOSUITE_TIMEOUT_SECONDS}-second process timeout and no test source was found."
            stage = "timeout"
        else:
            reason = "EvoSuite did not generate an *_ESTest.java file."
            stage = "generation"

        return {
            "source_row": source_row,
            "repetition": repetition,
            "status": "failed",
            "test_code": None,
            "test_count": 0,
            "elapsed_seconds": total_elapsed,
            "evosuite_seconds": evo_elapsed,
            "return_code": evo["return_code"],
            "error": reason,
            "stage": stage,
            "stdout_tail": text_tail(evo["stdout"]),
            "stderr_tail": text_tail(evo["stderr"]),
            "generated_files": [],
        }

    except Exception as exc:
        return make_failure_result(
            task, started_at, "exception", repr(exc)
        )

## 10. Run or resume all pending repetitions

The scheduler distributes work across Easy, Medium, and Hard. Each completed repetition is saved immediately. A worker exception leaves its task pending so it can be resumed in a later run.

In [ ]:
tasks = collect_pending_tasks()
print("=" * 78)
print("RESUMABLE EVOSUITE EXPERIMENT")
print("=" * 78)

for level in LEVELS:
    print(f"{level}: {sum(task['level'] == level for task in tasks)} pending")

print("Total pending:", len(tasks))
print("Maximum workers:", MAX_WORKERS)
print("Search budget:", SEARCH_BUDGET_SECONDS, "seconds")
print("EvoSuite process timeout:", EVOSUITE_TIMEOUT_SECONDS, "seconds")

completed_this_session = 0
worker_errors = []

if not tasks:
    print("No pending repetitions. Existing checkpoint results will be exported.")
else:
    worker_count = min(MAX_WORKERS, len(tasks))
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        future_to_task = {
            executor.submit(run_one_repetition, task): task
            for task in tasks
        }

        for future in as_completed(future_to_task):
            task = future_to_task[future]
            try:
                result = future.result()
                update_checkpoint(
                    level=task["level"],
                    source_row=result["source_row"],
                    repetition_no=result["repetition"],
                    result=result,
                )
                completed_this_session += 1
                print(
                    f"COMPLETED [{completed_this_session}/{len(tasks)}] | "
                    f"{task['level']} | Batch {task['batch']} | "
                    f"row={task['source_row']} | rep={task['repetition']} | "
                    f"{result['status']} | tests={result['test_count']} | "
                    f"time={result['elapsed_seconds']}s",
                    flush=True,
                )
            except Exception as exc:
                message = (
                    f"WORKER ERROR | {task['level']} | row={task['source_row']} | "
                    f"rep={task['repetition']} | {type(exc).__name__}: {exc}"
                )
                worker_errors.append(message)
                print(message, flush=True)

print("Completed during this session:", completed_this_session)
if worker_errors:
    print("Worker-level errors:")
    for message in worker_errors:
        print(message)

## 11. Checkpoint status

A `success` status means an EvoSuite test source file was found. A `failed` status means no generated test source was found or an earlier stage failed. Test compilation and execution are separate validation activities and are not included in this generation-success status.

In [ ]:
status_rows = []

for level in LEVELS:
    state = read_checkpoint(level)
    counts = {"success": 0, "failed": 0, "pending": 0}
    total_tests = 0

    for row in state["rows"]:
        for repetition in row["repetitions"].values():
            status = repetition.get("status", "pending")
            counts[status] = counts.get(status, 0) + 1
            total_tests += int(repetition.get("test_count") or 0)

    total_runs = state.get("total_runs", len(state["rows"]) * REPETITIONS_PER_PROJECT)
    status_rows.append({
        "level": level,
        "success": counts.get("success", 0),
        "failed": counts.get("failed", 0),
        "pending": counts.get("pending", 0),
        "total": total_runs,
        "generated_test_methods": total_tests,
    })

status_df = pd.DataFrame(status_rows)
print(status_df.to_string(index=False))
print("-" * 78)
print("ALL LEVELS")
print("Completed:", int(status_df["success"].sum() + status_df["failed"].sum()))
print("Pending:", int(status_df["pending"].sum()))
print("Total:", int(status_df["total"].sum()))

## 12. Export Excel, JSONL, raw checkpoints, and a ZIP archive

The checkpoint JSON and JSONL files preserve the complete generated source code. Excel cells are limited to 32,767 characters; unusually long source cells are marked as truncated in Excel, while the complete source remains available in the JSONL and checkpoint files.

In [ ]:
import math
from datetime import date, datetime

EXCEL_CELL_LIMIT = 32767


def json_safe_value(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
            return None
        return value
    if isinstance(value, (date, datetime)):
        return value.isoformat()
    if hasattr(value, "item"):
        try:
            return json_safe_value(value.item())
        except Exception:
            pass
    if isinstance(value, (list, tuple)):
        return [json_safe_value(item) for item in value]
    if isinstance(value, dict):
        return {str(key): json_safe_value(item) for key, item in value.items()}
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    return str(value)


def build_result_records():
    records = []

    for level in LEVELS:
        state = read_checkpoint(level)
        source_frame = data[level]

        for row_state in state["rows"]:
            source_row = int(row_state["source_row"])
            source_metadata = source_frame.iloc[source_row].to_dict()

            for repetition_key, repetition_result in row_state["repetitions"].items():
                record = {
                    **source_metadata,
                    "level": level,
                    "source_row": source_row,
                    "batch": int(row_state.get("batch", source_row + 1)),
                    "repetition": int(repetition_key),
                }
                record.update(repetition_result)
                records.append({key: json_safe_value(value) for key, value in record.items()})

    level_order = {level: index for index, level in enumerate(LEVELS)}
    records.sort(key=lambda item: (
        level_order.get(item["level"], 99),
        item["source_row"],
        item["repetition"],
    ))
    return records


def excel_safe_dataframe(frame):
    safe_frame = frame.copy()
    invalid_xml = re.compile(r"[\x00-\x08\x0B\x0C\x0E-\x1F]")

    for column in safe_frame.columns:
        if safe_frame[column].dtype == object:
            def clean_cell(value):
                if value is None:
                    return None
                if isinstance(value, (list, dict)):
                    value = json.dumps(value, ensure_ascii=False)
                if not isinstance(value, str):
                    return value
                value = invalid_xml.sub("", value)
                if len(value) > EXCEL_CELL_LIMIT:
                    suffix = "\n[TRUNCATED IN EXCEL; FULL CONTENT IS PRESERVED IN JSONL AND CHECKPOINTS]"
                    value = value[:EXCEL_CELL_LIMIT - len(suffix)] + suffix
                return value
            safe_frame[column] = safe_frame[column].map(clean_cell)

    return safe_frame


records = build_result_records()
results_df = pd.DataFrame(records)
summary_df = pd.DataFrame(status_rows)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
combined_excel = EXPORT_DIR / "EvoSuite_Java_150_Results.xlsx"
jsonl_path = EXPORT_DIR / "EvoSuite_Java_150_Results.jsonl"
failed_csv = EXPORT_DIR / "EvoSuite_Java_Failed_Runs.csv"
manifest_path = EXPORT_DIR / "manifest.json"

with open(jsonl_path, "w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

failed_df = results_df[results_df["status"] == "failed"].copy() if not results_df.empty else results_df.copy()
failed_df.to_csv(failed_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(combined_excel, engine="openpyxl") as writer:
    excel_safe_dataframe(results_df).to_excel(writer, sheet_name="All_Results", index=False)
    excel_safe_dataframe(summary_df).to_excel(writer, sheet_name="Summary", index=False)
    for level in LEVELS:
        level_frame = results_df[results_df["level"] == level].copy()
        excel_safe_dataframe(level_frame).to_excel(writer, sheet_name=level, index=False)

separate_excel_files = []
for level in LEVELS:
    level_frame = results_df[results_df["level"] == level].copy()
    level_summary = summary_df[summary_df["level"] == level].copy()
    level_path = EXPORT_DIR / f"EvoSuite_{level}_Results.xlsx"

    with pd.ExcelWriter(level_path, engine="openpyxl") as writer:
        excel_safe_dataframe(level_frame).to_excel(writer, sheet_name="Results", index=False)
        excel_safe_dataframe(level_summary).to_excel(writer, sheet_name="Summary", index=False)

    separate_excel_files.append(level_path)

checkpoint_export_dir = EXPORT_DIR / "checkpoints"
if checkpoint_export_dir.exists():
    shutil.rmtree(checkpoint_export_dir)

for level in LEVELS:
    source_checkpoint = BASE / level / "checkpoints" / "state.json"
    target_checkpoint = checkpoint_export_dir / level / "state.json"
    target_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_checkpoint, target_checkpoint)

manifest = {
    "experiment": "Defects4J and EvoSuite Java test generation",
    "levels": LEVELS,
    "rows_per_level": EXPECTED_ROWS_PER_LEVEL,
    "repetitions_per_project": REPETITIONS_PER_PROJECT,
    "expected_total_runs": EXPECTED_ROWS_PER_LEVEL * len(LEVELS) * REPETITIONS_PER_PROJECT,
    "search_budget_seconds": SEARCH_BUDGET_SECONDS,
    "max_workers": MAX_WORKERS,
    "evosuite_process_timeout_seconds": EVOSUITE_TIMEOUT_SECONDS,
    "evosuite_jar": str(EVOSUITE_JAR),
    "defects4j_path": str(D4J_PATH),
    "success_definition": "At least one *_ESTest.java file was generated. Tests were not compiled or executed as part of this success flag.",
    "summary": status_rows,
}
with open(manifest_path, "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

zip_path = Path("/content/EvoSuite_Java_150_Complete_Results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(combined_excel, arcname=combined_excel.name)
    archive.write(jsonl_path, arcname=jsonl_path.name)
    archive.write(failed_csv, arcname=failed_csv.name)
    archive.write(manifest_path, arcname=manifest_path.name)

    for level_path in separate_excel_files:
        archive.write(level_path, arcname=level_path.name)

    for checkpoint_file in checkpoint_export_dir.rglob("state.json"):
        archive.write(
            checkpoint_file,
            arcname=str(checkpoint_file.relative_to(EXPORT_DIR)),
        )

print("Export complete.")
print("Combined workbook:", combined_excel)
print("Full JSONL:", jsonl_path)
print("Failed runs:", failed_csv)
print("ZIP archive:", zip_path)
print("Total result rows:", len(results_df))
print(summary_df.to_string(index=False))

## 13. Download the complete results archive

The ZIP contains the combined workbook, three level-specific workbooks, the full JSONL export, failed-run details, the manifest, and raw checkpoints.

In [ ]:
from google.colab import files

if not zip_path.exists():
    raise FileNotFoundError(f"Results archive not found: {zip_path}")

print(f"Downloading {zip_path.name} ({zip_path.stat().st_size:,} bytes)")
files.download(str(zip_path))